## Notebook for developing clustering functionality

In [1]:
import os
import sys
import pandas as pd

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)
import plotting
import utils

In [7]:
base_data_folder = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
folder_name = "elec_s_37_ES_PT_no_bat_limit"
data_folder = os.path.join(base_data_folder, folder_name)

In [8]:
input_data = utils.load_csv_files_from_folder(data_folder)
batteries = input_data["batteries"]
branches = input_data["branches"]
generators = input_data["generators"]
capacity_factors = input_data["capacity_factors"]
generator_costs = input_data["generator_costs"]
hourly_demand = input_data["hourly_demand"]
nodes = input_data["nodes"]
nodes

,x,y,country
bus,,,
ES1 0,-3.427610,40.601332,ES
PT1 0,-8.282125,40.313466,PT


In [12]:
capacity_factors

,ES1 0 offwind-ac,ES1 0 onwind,ES1 0 ror,ES1 0 solar,PT1 0 offwind-ac,PT1 0 onwind,PT1 0 ror,PT1 0 solar,ES1 0 CCGT,PT1 0 CCGT,ES1 0 coal
snapshot,,,,,,,,,,,
2013-01-01 00:00:00,0.169080,0.198307,0.223631,0.0,0.223462,0.140790,0.108399,0.0,1.0,1.0,1.0
2013-01-01 01:00:00,0.180396,0.183442,0.209677,0.0,0.242486,0.109833,0.104136,0.0,1.0,1.0,1.0
2013-01-01 02:00:00,0.187817,0.173403,0.198885,0.0,0.251477,0.101679,0.100124,0.0,1.0,1.0,1.0
2013-01-01 03:00:00,0.198432,0.160254,0.187313,0.0,0.246618,0.098385,0.094812,0.0,1.0,1.0,1.0
2013-01-01 04:00:00,0.218048,0.156526,0.179705,0.0,0.247457,0.103719,0.089900,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2013-12-31 19:00:00,0.177343,0.219160,0.300193,0.0,0.196510,0.116910,0.636055,0.0,1.0,1.0,1.0
2013-12-31 20:00:00,0.181897,0.228008,0.288119,0.0,0.187475,0.109712,0.621952,0.0,1.0,1.0,1.0
2013-12-31 21:00:00,0.190584,0.231160,0.279827,0.0,0.172794,0.102505,0.611703,0.0,1.0,1.0,1.0


In [ ]:
import pandas as pd


def subset_by_weeks(df, year, weeks):
    """
    Returns a subset of the dataframe for specific ISO weeks of a given year.

    Args:
        df (pd.DataFrame): DataFrame with a DateTimeIndex.
        year (int): The year to filter by.
        weeks (list or set): A collection of ISO week numbers to include.

    Returns:
        pd.DataFrame: DataFrame subset for the specified weeks.
    """
    # Create a mask that filters by year and if the week number is in the provided weeks list
    week_mask = (df.index.year == year) & (
        df.index.to_series().dt.isocalendar().week.isin(weeks)
    )
    return df[week_mask]


def subset_by_months(df, year, months):
    """
    Returns a subset of the dataframe for specific months of a given year.

    Args:
        df (pd.DataFrame): DataFrame with a DateTimeIndex.
        year (int): The year to filter by.
        months (list or set): A collection of month numbers (1-12) to include.

    Returns:
        pd.DataFrame: DataFrame subset for the specified months.
    """
    # Create a mask that filters by year and if the month is in the provided months list
    month_mask = (df.index.year == year) & (df.index.month.isin(months))
    return df[month_mask]


# Example usage:
# Assuming 'capacity_factors' is your DataFrame with a datetime index

# To get data from the 10th and 12th weeks of 2013:
subset_weeks = subset_by_weeks(capacity_factors, 2013, [10, 12])
print("Subset by weeks:\n", subset_weeks)



Subset by weeks:
                      ES1 0 offwind-ac  ES1 0 onwind  ES1 0 ror  ES1 0 solar  \
snapshot                                                                      
2013-03-04 00:00:00          0.479367      0.258046   0.228923          0.0   
2013-03-04 01:00:00          0.490981      0.276300   0.228807          0.0   
2013-03-04 02:00:00          0.501064      0.290657   0.228692          0.0   
2013-03-04 03:00:00          0.500001      0.318964   0.228575          0.0   
2013-03-04 04:00:00          0.508508      0.350982   0.228466          0.0   
...                               ...           ...        ...          ...   
2013-03-24 19:00:00          0.476253      0.338240   0.370053          0.0   
2013-03-24 20:00:00          0.438930      0.333145   0.368712          0.0   
2013-03-24 21:00:00          0.396906      0.339499   0.367070          0.0   
2013-03-24 22:00:00          0.373620      0.352016   0.365366          0.0   
2013-03-24 23:00:00          0.371

In [16]:
def subset_by_weeks_with_index(df, year, weeks):
    """
    Returns two DataFrames:
      - The first DataFrame is the subset of the original DataFrame for the specified weeks.
      - The second DataFrame shows the snapshot dates for each week, with columns named
        like "week 3", "week 4", etc., and each cell in the column representing a date included.

    Args:
        df (pd.DataFrame): DataFrame with a DateTimeIndex.
        year (int): The year to filter by.
        weeks (list or set): A collection of ISO week numbers to include.

    Returns:
        tuple: (subset_df, index_df)
            subset_df: DataFrame subset for the specified weeks.
            index_df: DataFrame with each column representing a week and rows the snapshot dates.
    """
    # Get the subset of data for the specified weeks.
    subset_df = subset_by_weeks(df, year, weeks)

    # Create a dictionary where keys are the week labels and values are lists of dates.
    week_dates = {}
    for week in weeks:
        mask = (df.index.year == year) & (
            df.index.to_series().dt.isocalendar().week == week
        )
        dates = df.index[mask].tolist()
        week_dates[f"week {week}"] = dates

    # Convert the dictionary to a DataFrame. Lists may have different lengths, so we use pd.Series.
    index_df = pd.DataFrame(
        {week: pd.Series(dates) for week, dates in week_dates.items()}
    )

    return subset_df, index_df

In [19]:
weeks_to_include = [3, 7]
subset_data, index_data = subset_by_weeks_with_index(
    capacity_factors, 2013, weeks_to_include
)

In [38]:
weeks = [3 + 52 / 4 * i for i in range(4)]
weeks

[3.0, 16.0, 29.0, 42.0]

In [26]:
capacity_factors_copy = capacity_factors.copy()

In [28]:
capacity_factors_copy["week"] = (
    capacity_factors_copy.index.to_series().dt.isocalendar().week
)

In [45]:
capacity_factors_copy["month"] = capacity_factors_copy.index.to_series().dt.month

In [48]:
capacity_factors_copy["month"].value_counts()

month
1     744
3     744
5     744
7     744
8     744
10    744
12    744
4     720
6     720
9     720
11    720
2     672
Name: count, dtype: int64

In [50]:
df = subset_by_weeks(capacity_factors_copy, 2013, weeks)
df["week"].value_counts()

week
3     168
16    168
29    168
42    168
Name: count, dtype: Int64

In [51]:
months = [1, 4, 7, 10]
df = subset_by_months(capacity_factors_copy, 2013, months)
df["month"].value_counts()

month
1     744
7     744
10    744
4     720
Name: count, dtype: int64